# Carry Complexity Experiments

Hypothesis: diagonal mass concentration predicts carry complexity. When raw anti-diagonal mass is concentrated into a few degree positions, carry normalization has more excess mass to transport along the degree axis.

In [ ]:
from pathlib import Path
import sys

sys.path.append("../src")
if (Path.cwd() / "src").exists():
    sys.path.append(str(Path.cwd() / "src"))
elif (Path.cwd().parent / "src").exists():
    sys.path.append(str(Path.cwd().parent / "src"))

import matplotlib.pyplot as plt
import pandas as pd
from experiments import experiment_carry_complexity

In [ ]:
RESULTS = Path("results") if (Path.cwd() / "results").exists() else Path("../results")
RESULTS.mkdir(parents=True, exist_ok=True)

frames = []
for density in [0.2, 0.5, 1.0]:
    frame = experiment_carry_complexity(
        n_trials=60,
        num_digits=10,
        B=10,
        density=density,
        output_csv=str(RESULTS / f"carry_complexity_density_{density}.csv"),
    )
    frames.append(frame)

data = pd.concat(frames, ignore_index=True)
data.head()

In [ ]:
data.groupby("density")[[
    "diagonal_mass_concentration",
    "diagonal_mass_entropy",
    "carry_mass",
    "carry_entropy",
    "carry_amplification_ratio",
]].mean()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for density, group in data.groupby("density"):
    axes[0].scatter(
        group["diagonal_mass_concentration"],
        group["carry_mass"],
        alpha=0.65,
        label=f"density={density}",
    )
axes[0].set_xlabel("diagonal_mass_concentration")
axes[0].set_ylabel("carry_mass")
axes[0].set_title("Concentration vs carry mass")
axes[0].legend()

for density, group in data.groupby("density"):
    axes[1].scatter(
        group["diagonal_mass_entropy"],
        group["carry_entropy"],
        alpha=0.65,
        label=f"density={density}",
    )
axes[1].set_xlabel("diagonal_mass_entropy")
axes[1].set_ylabel("carry_entropy")
axes[1].set_title("Raw entropy vs carry entropy")

means = data.groupby("density")["carry_amplification_ratio"].mean()
axes[2].plot(means.index, means.values, marker="o")
axes[2].set_xlabel("density")
axes[2].set_ylabel("mean carry_amplification_ratio")
axes[2].set_title("Density vs amplification")

for ax in axes:
    ax.grid(alpha=0.25)

plt.tight_layout()
plt.show()

These plots are empirical probes rather than final laws. The working idea is that diagonal projection creates a raw mass distribution, and carry complexity records the nonlinear cost of turning that raw distribution into valid base-`B` digits.